In [1]:
# import os
# import random #데이터 샘플링
# from collections import Counter # count 용도

import numpy as np
import pandas as pd

# from geopy import distance # 거리 계산
# import geopy.distance
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

# 시각화
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
plt.style.use('fivethirtyeight')

# 한글, 마이너스 깨짐 방지
from matplotlib import rc, font_manager, rcParams
font=font_manager.FontProperties(fname="c:/Windows/Fonts/malgun.ttf").get_name()
rc('font', family = font)
rcParams['axes.unicode_minus'] = False

## 데이터 확인하기

In [2]:
df = pd.read_csv('./dataset/2019 관악구 업무추진비.csv')
df.head()

,연번,사용자,사용일시,사용시간,사용금액,사용장소,주소,사용방법,사용내역,집행대상
0,1,의장,2019-01-02,08:44,58500,완산정,서울 관악구 봉천로 484,신용카드,의정활동 및 의회운영 관련 업무휴대경비,의장 등 9명
1,2,의장,2019-01-02,12:36,89000,제주은갈치,서울 관악구 관악로 139,신용카드,의정활동 및 의회운영 관련 업무휴대경비,의장 등 7명
2,3,의장,2019-01-04,12:53,14000,아리차이,서울 관악구 신림동길 4,신용카드,의정활동 및 의회운영 관련 업무휴대경비,의장 등 2명
3,4,의장,2019-01-09,12:33,77000,남원추어탕,서울 관악구 쑥고개로 135,신용카드,의정활동 및 의회운영 관련 업무휴대경비,의장 등 7명
4,5,의장,2019-01-09,14:48,83300,카페베네,서울 관악구 남부순환로 2082-29,신용카드,의정활동 및 의회운영 관련 업무휴대경비,의장 등 17명


In [3]:
df.tail()

,연번,사용자,사용일시,사용시간,사용금액,사용장소,주소,사용방법,사용내역,집행대상
1999,101,의회사무국,2019-12-27,20:27,132000,시골집,서울 관악구 낙성대로 22-7,신용카드,의정업무 추진관계자 간담회,사무국장 등 10명
2000,102,의회사무국,2019-12-30,13:00,220000,만리장성,서울 관악구 관악로 146,신용카드,사무국 직원 격려,의정팀장 등 10명
2001,103,의회사무국,2019-12-30,13:08,87000,쌈마을,서울 관악구 남부순환로226길 31,신용카드,의사팀 업무추진 관련 간담회비,의사팀 직원 10명
2002,104,의회사무국,2019-12-30,14:01,174000,장보고마트,서울 관악구 남부순환로230길,신용카드,부서운영 물품(음료) 구입,"의원, 직원, 방문민원인 등"
2003,105,의회사무국,2019-12-30,20:24,123000,제주은갈치,서울 관악구 관악로 139,신용카드,신년인사회 추진 관련 관계자 간담회,의정팀장 등 7명


In [4]:
df.shape

(2004, 10)

In [5]:
df.columns

Index(['연번', '사용자', '사용일시', '사용시간', '사용금액', '사용장소', '주소', '사용방법', '사용내역',
       '집행대상'],
      dtype='object')

In [6]:
df.isnull().sum()

연번        0
사용자       0
사용일시      0
사용시간    452
사용금액      0
사용장소     11
주소       11
사용방법      0
사용내역      0
집행대상     28
dtype: int64

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2004 entries, 0 to 2003
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   연번      2004 non-null   int64 
 1   사용자     2004 non-null   object
 2   사용일시    2004 non-null   object
 3   사용시간    1552 non-null   object
 4   사용금액    2004 non-null   int64 
 5   사용장소    1993 non-null   object
 6   주소      1993 non-null   object
 7   사용방법    2004 non-null   object
 8   사용내역    2004 non-null   object
 9   집행대상    1976 non-null   object
dtypes: int64(2), object(8)
memory usage: 156.7+ KB


In [8]:
df.sum()

연번                                                  88964
사용자     의장의장의장의장의장의장의장의장의장의장의장의장의장의장의장의장의장의장의장의장의장의장의장...
사용일시    2019-01-022019-01-022019-01-042019-01-092019-0...
사용금액                                            224074380
사용방법    신용카드신용카드신용카드신용카드신용카드신용카드신용카드신용카드현금신용카드신용카드신용카드...
사용내역    의정활동 및 의회운영 관련 업무휴대경비의정활동 및 의회운영 관련 업무휴대경비의정활동...
dtype: object

In [9]:
df['사용장소'].nunique()

611

In [10]:
df['사용내역'].nunique()

213

+ 총 사용금액 224,074,380 (2억2천~)
+ 장소 611곳

+ 필요한 것
    - 필요없는 컬럼 정리
    - 컬럼 타입 변경
    - 음식점인지 아닌지 구별하기
    - 음식 카테고리 추가하기

## 데이터 전처리하기

+ 필요없는 컬럼 삭제
+ 컬럼 데이터 타입 변경

In [11]:
# 사용방법 컬럼 확인
df['사용방법'].unique()

array(['신용카드', '현금', '제로페이'], dtype=object)

In [12]:
# 사용방법 컬럼 확인
df['사용자'].unique()

array(['의장', '부의장', '의회운영위원장', '행정재경위원장', '보건복지위원장', '도시건설위원장', '관악구의회',
       '의회사무국'], dtype=object)

In [13]:
df

,연번,사용자,사용일시,사용시간,사용금액,사용장소,주소,사용방법,사용내역,집행대상
0,1,의장,2019-01-02,08:44,58500,완산정,서울 관악구 봉천로 484,신용카드,의정활동 및 의회운영 관련 업무휴대경비,의장 등 9명
1,2,의장,2019-01-02,12:36,89000,제주은갈치,서울 관악구 관악로 139,신용카드,의정활동 및 의회운영 관련 업무휴대경비,의장 등 7명
2,3,의장,2019-01-04,12:53,14000,아리차이,서울 관악구 신림동길 4,신용카드,의정활동 및 의회운영 관련 업무휴대경비,의장 등 2명
3,4,의장,2019-01-09,12:33,77000,남원추어탕,서울 관악구 쑥고개로 135,신용카드,의정활동 및 의회운영 관련 업무휴대경비,의장 등 7명
4,5,의장,2019-01-09,14:48,83300,카페베네,서울 관악구 남부순환로 2082-29,신용카드,의정활동 및 의회운영 관련 업무휴대경비,의장 등 17명
...,...,...,...,...,...,...,...,...,...,...
1999,101,의회사무국,2019-12-27,20:27,132000,시골집,서울 관악구 낙성대로 22-7,신용카드,의정업무 추진관계자 간담회,사무국장 등 10명
2000,102,의회사무국,2019-12-30,13:00,220000,만리장성,서울 관악구 관악로 146,신용카드,사무국 직원 격려,의정팀장 등 10명
2001,103,의회사무국,2019-12-30,13:08,87000,쌈마을,서울 관악구 남부순환로226길 31,신용카드,의사팀 업무추진 관련 간담회비,의사팀 직원 10명
2002,104,의회사무국,2019-12-30,14:01,174000,장보고마트,서울 관악구 남부순환로230길,신용카드,부서운영 물품(음료) 구입,"의원, 직원, 방문민원인 등"


In [14]:
# 연번, 사용방법, 집행대상 컬럼 삭제
df = df.drop(['연번', '사용방법', '집행대상'], axis=1)
df.head()

,사용자,사용일시,사용시간,사용금액,사용장소,주소,사용내역
0,의장,2019-01-02,08:44,58500,완산정,서울 관악구 봉천로 484,의정활동 및 의회운영 관련 업무휴대경비
1,의장,2019-01-02,12:36,89000,제주은갈치,서울 관악구 관악로 139,의정활동 및 의회운영 관련 업무휴대경비
2,의장,2019-01-04,12:53,14000,아리차이,서울 관악구 신림동길 4,의정활동 및 의회운영 관련 업무휴대경비
3,의장,2019-01-09,12:33,77000,남원추어탕,서울 관악구 쑥고개로 135,의정활동 및 의회운영 관련 업무휴대경비
4,의장,2019-01-09,14:48,83300,카페베네,서울 관악구 남부순환로 2082-29,의정활동 및 의회운영 관련 업무휴대경비


In [15]:
# 사용일시 컬럼 날짜타입으로 변경
df['사용일시'] = pd.to_datetime(df['사용일시'])

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2004 entries, 0 to 2003
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   사용자     2004 non-null   object        
 1   사용일시    2004 non-null   datetime64[ns]
 2   사용시간    1552 non-null   object        
 3   사용금액    2004 non-null   int64         
 4   사용장소    1993 non-null   object        
 5   주소      1993 non-null   object        
 6   사용내역    2004 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(5)
memory usage: 109.7+ KB


In [17]:
df['사용장소'].value_counts()[0]

93

In [18]:
for i in df.groupby('사용장소').sum().index:
    print(i)

#207
(주)소명생고기
(주)송원마포갈비
(주)신세계페이먼츠
(주)신화아이푸드
(주)영풍안성상휴게
(주)이마트
11번가
406CAFE
ENZO(엔조)
GS슈퍼
GS슈퍼마켓
KFC
KSNET
SEROTONIN
SOME801
㈜배다리문화원
가마솥
가마솥삼계탕
가빈
가야촌오리주물럭
가야촌유황오리
가온매운갈비찜
가츠몽
감격시대
감나무집
값진식육
강강술래
강강술래신림본동점
강남식당
강문어화횟집
강촌민물매운탕2호점
강촌민물매운탕２호점
개성보쌈
갯바위
갯벌낙지
거목쟁반짜장
경복궁
경식이네알쌈쭈꾸미
고기만족
고기킹
고려왕족발
고봉민김밥
고봉민김밥인
고창복의낙지세상
고향순대국
고향식당
고흥식당
광명수산
광어삼촌
교대전집
교동왕족발
교촌치킨
구름산원주추어탕
구수옥설렁탕
국내산생고기
국회루
굴러들어온복
그린파크식당
금비
금수강산
금수사
기사님식당
기장산곰장어포차
기절초풍왕순대
김가네
김가네 관악구청점
김밥
김밥천국
김밥＆백반
김용해순대국
김채원24시감자탕
깡돈
꽃올림
꿀벌식당
나의집
나주곰탕
나주오리종가
낙성푸줏간
낙지마을해물구이
낙지촌
난곡문성파리바게뜨
난곡사거리 파리바게뜨
남궁야
남도식당
남도전문음식점
남도해물요리전문점
남원추어탕
낭만모로코
내자리에프씨카페
너구리칼국수
노가리7080
농장사람들
다락방
다리목
다빈
다온죽카페
다우데이타
다이소
달구리
당곡기사식당
당진강완옥아귀찜
당진아구동태찜탕
대가한방오리
대박수산
대주직판장
대호아구집
더고기서울대맛집
더블미트 봉천점
더진국
던킨도너츠
도일각
돈가대박집
돌체
동강염소탕
동강영양탕
동경산책
동원각
동태찜&코다리냉면
동해
동해물회
동해세꼬시
동해수산
동해안횟집
두꺼비맛집
두꺼비황소곱창
두부이야기
두부이야기봉천점
들향기칼국수쭈꾸미
등갈비연탄구이
등촌샤브칼국수
딸랏롯빠이
떡보의하루
또와봐순대
뚱띵이곱창
뚱띵이왕갈비
라온
라티놀
라페리체
란
런던부페
려
롯데리아（백령도점）
리몽
마노아숯불구이
마늘오리
마당쇠
마야
마을부엌어울림
마켓동광
마포소금구이
마포숯불갈비
만년닭강정
만다린
만두와김밥
만리성
만리장성
만양순대국소머리국밥
맘스터치
맘

In [19]:
df[df['사용장소']=='가야촌유황오리']

,사용자,사용일시,사용시간,사용금액,사용장소,주소,사용내역
396,부의장,2019-05-04,18:49,71000,가야촌유황오리,서울 관악구 봉천로 407,의회운영 및 의정활동 관련 유대경비


In [20]:
# for i in df.index:
#     # GS슈퍼마켓
#     if df.loc[i, '사용장소']=='GS슈퍼':
#         df.loc[i, '사용장소'] = 'GS슈퍼마켓'
    
#     # 가마솥삼계탕
#     elif df.loc[i, '사용장소']=='가마솥삼계탕':
#         df.loc[i, '사용장소'] = '가마솥삼계탕 신림본점'
    
#     # 가마솥
#     elif df.loc[i, '사용장소']=='가마솥':
#         df.loc[i, '사용장소'] = '관악산가마솥삼계탕'

In [21]:
df_group_cnt = df['사용장소'].value_counts().reset_index()
df_group_cnt = df_group_cnt.sort_values('index')
df_group_cnt.index = range(611)
df_group_cnt

,index,사용장소
0,#207,2
1,(주)소명생고기,1
2,(주)송원마포갈비,1
3,(주)신세계페이먼츠,4
4,(주)신화아이푸드,1
...,...,...
606,흥부보쌈,8
607,희로애락,2
608,희전명가,1
609,히포커피,1


In [22]:
for i in df_group_cnt.index:
    print(df_group_cnt.loc[i, 'index'], df_group_cnt.loc[i, '사용장소'])

#207 2
(주)소명생고기 1
(주)송원마포갈비 1
(주)신세계페이먼츠 4
(주)신화아이푸드 1
(주)영풍안성상휴게 1
(주)이마트 1
11번가 4
406CAFE 1
ENZO(엔조) 1
GS슈퍼 2
GS슈퍼마켓 2
KFC 1
KSNET 1
SEROTONIN 1
SOME801 1
㈜배다리문화원 1
가마솥 4
가마솥삼계탕 1
가빈 2
가야촌오리주물럭 1
가야촌유황오리 1
가온매운갈비찜 1
가츠몽 1
감격시대 1
감나무집 33
값진식육 1
강강술래 14
강강술래신림본동점 7
강남식당 1
강문어화횟집 1
강촌민물매운탕2호점 6
강촌민물매운탕２호점 1
개성보쌈 1
갯바위 6
갯벌낙지 1
거목쟁반짜장 8
경복궁 1
경식이네알쌈쭈꾸미 1
고기만족 1
고기킹 1
고려왕족발 5
고봉민김밥 1
고봉민김밥인 2
고창복의낙지세상 11
고향순대국 1
고향식당 2
고흥식당 3
광명수산 2
광어삼촌 2
교대전집 1
교동왕족발 1
교촌치킨 2
구름산원주추어탕 1
구수옥설렁탕 9
국내산생고기 1
국회루 2
굴러들어온복 1
그린파크식당 5
금비 12
금수강산 2
금수사 2
기사님식당 1
기장산곰장어포차 1
기절초풍왕순대 4
김가네 7
김가네 관악구청점 2
김밥 5
김밥천국 6
김밥＆백반 1
김용해순대국 2
김채원24시감자탕 1
깡돈 1
꽃올림 1
꿀벌식당 1
나의집 1
나주곰탕 12
나주오리종가 4
낙성푸줏간 1
낙지마을해물구이 1
낙지촌 1
난곡문성파리바게뜨 1
난곡사거리 파리바게뜨 1
남궁야 1
남도식당 1
남도전문음식점 5
남도해물요리전문점 3
남원추어탕 33
낭만모로코 1
내자리에프씨카페 1
너구리칼국수 1
노가리7080 1
농장사람들 2
다락방 1
다리목 12
다빈 1
다온죽카페 1
다우데이타 1
다이소 8
달구리 1
당곡기사식당 11
당진강완옥아귀찜 1
당진아구동태찜탕 2
대가한방오리 1
대박수산 1
대주직판장 1
대호아구집 7
더고기서울대맛집 1
더블미트 봉천점 1
더진국 1
던킨도너츠 2
도일각 1
돈가대박집 1
돌체 1
동강염소탕 3
동강영양탕 1
동경산책 1
동원각 1
동

In [23]:
for i in df['사용장소'].value_counts().reset_index().index:
    print(df['사용장소'].value_counts().reset_index().loc[i])

index    제주은갈치
사용장소        93
Name: 0, dtype: object
index    시골집
사용장소      45
Name: 1, dtype: object
index    옥천골
사용장소      40
Name: 2, dtype: object
index    남원추어탕
사용장소        33
Name: 3, dtype: object
index    감나무집
사용장소       33
Name: 4, dtype: object
index    오리오리
사용장소       31
Name: 5, dtype: object
index    산야로
사용장소      31
Name: 6, dtype: object
index    쌈마을
사용장소      26
Name: 7, dtype: object
index    옛날육개장
사용장소        26
Name: 8, dtype: object
index    장보고마트
사용장소        25
Name: 9, dtype: object
index    하이보
사용장소      23
Name: 10, dtype: object
index    떡보의하루
사용장소        22
Name: 11, dtype: object
index    우리동네고기집
사용장소          22
Name: 12, dtype: object
index    만리장성
사용장소       21
Name: 13, dtype: object
index    생각보다맛있는집
사용장소           21
Name: 14, dtype: object
index    청능장
사용장소      19
Name: 15, dtype: object
index    은성숯불갈비
사용장소         19
Name: 16, dtype: object
index    콩심
사용장소     19
Name: 17, dtype: object
index    어메칼국수
사용장소        18
Name: 18, dtype: object
index   

index    하우림
사용장소       3
Name: 163, dtype: object
index    부산복집
사용장소        3
Name: 164, dtype: object
index    와사비
사용장소       3
Name: 165, dtype: object
index    종로빈대떡
사용장소         3
Name: 166, dtype: object
index    원주추어탕
사용장소         2
Name: 167, dtype: object
index    GS슈퍼
사용장소        2
Name: 168, dtype: object
index    위메프
사용장소       2
Name: 169, dtype: object
index    커피빈코리아
사용장소          2
Name: 170, dtype: object
index    장수생삼겹살
사용장소          2
Name: 171, dtype: object
index    커피더알
사용장소        2
Name: 172, dtype: object
index    안씨네쭈꾸미
사용장소          2
Name: 173, dtype: object
index    참이맛감자탕
사용장소          2
Name: 174, dtype: object
index    본죽＆비빔밥ｃａｆｅ
사용장소              2
Name: 175, dtype: object
index    콩심서울대점
사용장소          2
Name: 176, dtype: object
index    일커피 서울대입구점
사용장소              2
Name: 177, dtype: object
index    목포산낙지철판해물
사용장소             2
Name: 178, dtype: object
index    보현푸드
사용장소        2
Name: 179, dtype: object
index    동해안횟집
사용장소         2
Name: 180, dtype:

index    삐에스몽테과자점
사용장소            1
Name: 339, dtype: object
index    바우네나주곰탕　봉천
사용장소              1
Name: 340, dtype: object
index    진미별미순대국
사용장소           1
Name: 341, dtype: object
index    경복궁
사용장소       1
Name: 342, dtype: object
index    어물전
사용장소       1
Name: 343, dtype: object
index    만양순대국소머리국밥
사용장소              1
Name: 344, dtype: object
index    중국성
사용장소       1
Name: 345, dtype: object
index    본죽&본도시락
사용장소           1
Name: 346, dtype: object
index    리몽
사용장소      1
Name: 347, dtype: object
index    명품청기와감자탕미성점
사용장소               1
Name: 348, dtype: object
index    춘천골숯불닭갈비
사용장소            1
Name: 349, dtype: object
index    테라스
사용장소       1
Name: 350, dtype: object
index    고향순대국
사용장소         1
Name: 351, dtype: object
index    원조정가네매운갈비찜
사용장소              1
Name: 352, dtype: object
index    마늘오리
사용장소        1
Name: 353, dtype: object
index    돌체
사용장소      1
Name: 354, dtype: object
index    어회가
사용장소       1
Name: 355, dtype: object
index    옛터
사용장소      1
Name: 356, dt

index    훌랄라치킨
사용장소         1
Name: 488, dtype: object
index    삼다돈가(봉천점)
사용장소             1
Name: 489, dtype: object
index    봉이 구이대장
사용장소           1
Name: 490, dtype: object
index    영암팥칼국수
사용장소          1
Name: 491, dtype: object
index    정관장
사용장소       1
Name: 492, dtype: object
index    마포소금구이
사용장소          1
Name: 493, dtype: object
index    아웃백
사용장소       1
Name: 494, dtype: object
index    이태리부대찌개
사용장소           1
Name: 495, dtype: object
index    베어문
사용장소       1
Name: 496, dtype: object
index    칠분도스시
사용장소         1
Name: 497, dtype: object
index    토방집등심생고기
사용장소            1
Name: 498, dtype: object
index    SOME801
사용장소           1
Name: 499, dtype: object
index    탐앤탐스
사용장소        1
Name: 500, dtype: object
index    칼국수한마당
사용장소          1
Name: 501, dtype: object
index    마포숯불갈비
사용장소          1
Name: 502, dtype: object
index    미소야
사용장소       1
Name: 503, dtype: object
index    애슐리서울대입구점
사용장소             1
Name: 504, dtype: object
index    카페１번가
사용장소         1
Name: 505, 

In [24]:
df_group_cnt[df_group_cnt['사용장소']>4]

,index,사용장소
25,감나무집,33
27,강강술래,14
28,강강술래신림본동점,7
31,강촌민물매운탕2호점,6
34,갯바위,6
...,...,...
581,할매보쌈,5
583,해물나라,17
591,홍콩반점,7
593,황복,6


In [25]:
df['사용장소'].value_counts().head(20)

제주은갈치       93
시골집         45
옥천골         40
남원추어탕       33
감나무집        33
오리오리        31
산야로         31
쌈마을         26
옛날육개장       26
장보고마트       25
하이보         23
떡보의하루       22
우리동네고기집     22
만리장성        21
생각보다맛있는집    21
청능장         19
은성숯불갈비      19
콩심          19
어메칼국수       18
해물나라        17
Name: 사용장소, dtype: int64

In [26]:
df.groupby('사용장소').sum().sort_values('사용금액', ascending=False).head(20)

,사용금액
사용장소,
제주은갈치,11714000
시골집,9381000
우리동네고기집,7319900
옥천골,6801000
감나무집,6527500
장보고마트,6522210
쿠팡,4728660
하이보,4470000
쌈마을,4350000


## 사용자별 데이터 확인

In [27]:
df['사용자'].unique()

array(['의장', '부의장', '의회운영위원장', '행정재경위원장', '보건복지위원장', '도시건설위원장', '관악구의회',
       '의회사무국'], dtype=object)

### 의장

In [28]:
df[df['사용자']=='의장'].groupby('사용장소').count().sort_values('사용자', ascending=False).head(25)

,사용자,사용일시,사용시간,사용금액,주소,사용내역
사용장소,,,,,,
제주은갈치,33,33,33,33,33,33
우리동네고기집,22,22,22,22,22,22
감나무집,18,18,18,18,18,18
생각보다맛있는집,16,16,16,16,16,16
남원추어탕,15,15,15,15,15,15
떡보의하루,11,11,11,11,11,11
옛날육개장,9,9,9,9,9,9
시골집,9,9,9,9,9,9
정가,7,7,7,7,7,7


### 부의장

In [29]:
df[df['사용자']=='부의장'].groupby('사용장소').count().sort_values('사용자', ascending=False).head(25)

,사용자,사용일시,사용시간,사용금액,주소,사용내역
사용장소,,,,,,
은성숯불갈비,19,19,19,19,19,19
당곡기사식당,11,11,11,11,11,11
고창복의낙지세상,10,10,10,10,10,10
구수옥설렁탕,9,9,9,9,9,9
아리차이,7,7,7,7,7,7
숨은집,7,7,7,7,7,7
해물나라,6,6,6,6,6,6
황복,6,6,6,6,6,6
제주은갈치,5,5,5,5,5,5


### 의회운영위원장

In [30]:
df[df['사용자']=='의회운영위원장'].groupby('사용장소').count().sort_values('사용자', ascending=False).head(25)

,사용자,사용일시,사용시간,사용금액,주소,사용내역
사용장소,,,,,,
다리목,11,11,11,11,11,11
북경반점,7,7,7,7,7,7
김밥,5,5,5,5,5,5
명품횡성한우가,5,5,5,5,5,5
장터순대국,5,5,5,5,5,5
올림반찬,3,3,3,3,3,3
남원추어탕,3,3,3,3,3,3
작은따옴표협동조합,3,3,3,3,3,3
차이나영,3,3,3,3,3,3


### 행정재경위원장

In [31]:
df[df['사용자']=='행정재경위원장'].groupby('사용장소').count().sort_values('사용자', ascending=False).head(25)

,사용자,사용일시,사용시간,사용금액,주소,사용내역
사용장소,,,,,,
전진식당,10,10,10,10,10,10
통큰대박집,10,10,10,10,10,10
거목쟁반짜장,8,8,8,8,8,8
콩심,7,7,7,7,7,7
금비,6,6,6,6,6,6
안집,5,5,5,5,5,5
제주은갈치,5,5,5,5,5,5
송원마포갈비,4,4,4,4,4,4
등갈비연탄구이,4,4,4,4,4,4


### 보건복지위원장

In [32]:
df[df['사용자']=='보건복지위원장'].groupby('사용장소').count().sort_values('사용자', ascending=False).head(25)

,사용자,사용일시,사용시간,사용금액,주소,사용내역
사용장소,,,,,,
옥천골,37,37,37,37,37,37
제주은갈치,20,20,20,20,20,20
감나무집,6,6,6,6,6,6
정육점식당,5,5,5,5,5,5
목포부부아구찜,5,5,5,5,5,5
옛날육개장,3,3,3,3,3,3
콩심,3,3,3,3,3,3
유창숯불갈비,3,3,3,3,3,3
고흥식당,3,3,3,3,3,3


### 도시건설위원장

In [33]:
df[df['사용자']=='도시건설위원장'].groupby('사용장소').count().sort_values('사용자', ascending=False).head(25)

,사용자,사용일시,사용시간,사용금액,주소,사용내역
사용장소,,,,,,
감나무집,5,5,5,5,5,5
만리장성,5,5,5,5,5,5
흥부보쌈,5,5,5,5,5,5
사랑방,5,5,5,5,5,5
남도전문음식점,4,4,4,4,4,4
용궁,4,4,4,4,4,4
선우정,4,4,4,4,4,4
우성각,4,4,4,4,4,4
일구칠구매운탕,3,3,3,3,3,3


### 관악구의회

In [34]:
df[df['사용자']=='관악구의회'].groupby('사용장소').count().sort_values('사용자', ascending=False).head(25)

,사용자,사용일시,사용시간,사용금액,주소,사용내역
사용장소,,,,,,


In [35]:
df[df['사용자']=='관악구의회']

,사용자,사용일시,사용시간,사용금액,사용장소,주소,사용내역
370,관악구의회,2019-04-26,NaN,1200000,NaN,NaN,강원도 산불 피해지원 성금 납부


### 의회사무국

In [36]:
df[df['사용자']=='의회사무국'].groupby('사용장소').count().sort_values('사용자', ascending=False).head(25)

,사용자,사용일시,사용시간,사용금액,주소,사용내역
사용장소,,,,,,
시골집,33,33,17,33,33,33
제주은갈치,29,29,15,29,29,29
산야로,29,29,17,29,29,29
오리오리,29,29,9,29,29,29
장보고마트,25,25,15,25,25,25
어메칼국수,18,18,8,18,18,18
쌈마을,18,18,9,18,18,18
하이보,18,18,12,18,18,18
쿠팡,17,17,8,17,17,17
